In [1]:
# !pip install 'zarr<3'
# !pip install timm


In [2]:
# ALWAYS RUN THIS FIRST!
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Specify GPU 0 (out of 4 available GPUs)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

NOTEBOOK_DIR = Path("/rsrch9/home/plm/idso_fa1_pathology/codes/yshokrollahi/vitamin-p-latest")
os.chdir(NOTEBOOK_DIR)
sys.path.insert(0, str(NOTEBOOK_DIR))
print(f"✅ Working directory: {os.getcwd()}")
print(f"✅ Using GPU: {os.environ.get('CUDA_VISIBLE_DEVICES', 'Not set')}")

✅ Working directory: /rsrch9/home/plm/idso_fa1_pathology/codes/yshokrollahi/vitamin-p-latest
✅ Using GPU: 0


In [3]:
# Cell 3: Import and create dataloaders
from dataset import Config, create_dataloaders

# Just use the correct relative path from your working directory
config = Config("configs/training/config_fold33.yaml")  # Note: "configs" not "config"
config.print_config()

train_loader, val_loader, test_loader = create_dataloaders(config)
print("\n✅ Ready to use!")

✅ CRC Dataset Package v1.0.0 loaded
CRC DATASET CONFIGURATION
Config File: configs/training/config_fold33.yaml
Zarr Base: /rsrch9/home/plm/idso_fa1_pathology/TIER2/yasin-vitaminp/pannuke/zarr_data
Cache: Disabled
Strategy: memory

📊 Data Splits:
  Train: 2 samples
  Val: 1 samples
  Test: 1 samples

🔄 DataLoader:
  Batch Size: 4
  Num Workers: 16
  Pin Memory: True

🎨 Augmentation:
  Training: True
  Probability: 0.5

🎯 HV Maps:
  Generate: True
  Method: pannuke
  HE Nuclei: True
  HE Cells: True
  MIF Nuclei: True
  MIF Cells: True

🔍 Filtering:
  Min Instances: 0
  Filter Empty: True

CREATING DATALOADERS
Strategy: memory
Use Cache: False
Batch Size: 4
Num Workers: 16
Train split: 0 CRC + 0 Xenium + 0 TissueNet + 2 PanNuke + 0 Lizard + 0 MoNuSeg + 0 MoNuSAC + 0 TNBC + 0 NuInsSeg + 0 CryoNuSeg + 0 BC + 0 CoNSeP + 0 Kumar + 0 CPM17 + 0 CPM15 + 0 HE_DSB2018 + 0 Fluo_DSB2018 + 0 PanopTILs
Val   split: 0 CRC + 0 Xenium + 0 TissueNet + 1 PanNuke + 0 Lizard + 0 MoNuSeg + 0 MoNuSAC + 0 TNBC

## Flex model

In [4]:
import torch
import numpy as np
from collections import defaultdict
from tqdm import tqdm
from metrics import get_fast_pq, aggregated_jaccard_index
from vitaminp import VitaminPFlex, SimplePreprocessing, prepare_he_input, prepare_mif_input
from postprocessing import process_model_outputs

# ==========================================
# 1. CONFIGURATION: DATASET MODALITIES
# ==========================================
DATASET_MODALITIES = {

    # MIF Only Datasets
    'tissuenet': {'he_nuclei': False, 'he_cell': False,  'mif_nuclei': True,  'mif_cell': True},

    # Combined / Paired Datasets
    'crc':       {'he_nuclei': True,  'he_cell': True,   'mif_nuclei': True,  'mif_cell': True},
    # H&E Only Datasets
    'pannuke':   {'he_nuclei': True,  'he_cell': False,   'mif_nuclei': False, 'mif_cell': False},
    'lizard':    {'he_nuclei': True,  'he_cell': False,  'mif_nuclei': False, 'mif_cell': False},
    'monuseg':   {'he_nuclei': True,  'he_cell': False,  'mif_nuclei': False, 'mif_cell': False},
    'tnbc':      {'he_nuclei': True,  'he_cell': False,  'mif_nuclei': False, 'mif_cell': False},
    'nuinsseg':  {'he_nuclei': True,  'he_cell': False,  'mif_nuclei': False, 'mif_cell': False},
    'cryonuseg': {'he_nuclei': True,  'he_cell': False,  'mif_nuclei': False, 'mif_cell': False},
    'bc':        {'he_nuclei': True,  'he_cell': False,  'mif_nuclei': False, 'mif_cell': False},
    'consep':    {'he_nuclei': True,  'he_cell': False,  'mif_nuclei': False, 'mif_cell': False},
    'monusac':   {'he_nuclei': True,  'he_cell': False,  'mif_nuclei': False, 'mif_cell': False},
    'kumar':     {'he_nuclei': True,  'he_cell': False,  'mif_nuclei': False, 'mif_cell': False},
    'cpm17':     {'he_nuclei': True,  'he_cell': False,  'mif_nuclei': False, 'mif_cell': False},
    

}

# ==========================================
# 2. SETUP & TTA HELPERS
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

print("\n📦 Loading model...")
model = VitaminPFlex(model_size='large').to(device)
checkpoint_path = "checkpoints/vitamin_p_flex_large_fold33_pannuke_lessaug_best.pth"
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()
preprocessor = SimplePreprocessing()
print(f"✅ Model loaded")

# Structure: metrics[dataset][mode][metric_name] -> list
dataset_metrics = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
dataset_counts = defaultdict(int)

def get_tta_outputs(model, img_tensor):
    """
    Runs inference 3 times (Orig, HFlip, VFlip) and returns 
    both the Original output (Base) and the Averaged output (TTA).
    """
    # 1. Original
    out_orig = model(img_tensor)

    # 2. H-Flip
    img_h = torch.flip(img_tensor, [3])
    out_h = model(img_h)
    # De-augment H
    for k in out_h.keys():
        out_h[k] = torch.flip(out_h[k], [3])
        if 'hv' in k: out_h[k][:, 0, :, :] *= -1 

    # 3. V-Flip
    img_v = torch.flip(img_tensor, [2])
    out_v = model(img_v)
    # De-augment V
    for k in out_v.keys():
        out_v[k] = torch.flip(out_v[k], [2])
        if 'hv' in k: out_v[k][:, 1, :, :] *= -1 

    # Aggregate
    out_avg = {}
    for k in out_orig.keys():
        stack = torch.stack([out_orig[k], out_h[k], out_v[k]])
        out_avg[k] = torch.mean(stack, dim=0)
        
    return out_orig, out_avg

# ==========================================
# 3. EVALUATION LOOP
# ==========================================
print("\n🚀 Starting Global Evaluation (Comparing Base vs. TTA)...")

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Processing Test Set"):
        
        sources = batch.get('dataset_source', [])
        
        # --- Prepare Inputs ---
        
        # 1. H&E Input (Prepare if ANY dataset needs it, or just prepare always to be safe/simple)
        # Optimization: We can check if any source needs H&E, but for now we run it standard.
        he_img = prepare_he_input(batch['he_image'].to(device))
        he_img = preprocessor.percentile_normalize(he_img)
        
        # 2. MIF Input (Check if ANY source in this batch requires MIF)
        has_mif_in_batch = False
        for s in sources:
            if s in DATASET_MODALITIES:
                if DATASET_MODALITIES[s]['mif_nuclei'] or DATASET_MODALITIES[s]['mif_cell']:
                    has_mif_in_batch = True
                    break
        
        mif_img = None
        if has_mif_in_batch:
            mif_img = prepare_mif_input(batch['mif_image'].to(device))
            mif_img = preprocessor.percentile_normalize(mif_img)
        
        # --- Run Inference (Get BOTH Base and TTA) ---
        out_he_base, out_he_tta = get_tta_outputs(model, he_img)
        
        out_mif_base, out_mif_tta = (None, None), (None, None)
        if has_mif_in_batch:
            out_mif_base, out_mif_tta = get_tta_outputs(model, mif_img)

        # Iterate samples in batch
        batch_size = len(sources)
        for i in range(batch_size):
            ds_name = sources[i]
            if ds_name not in DATASET_MODALITIES: continue
                
            dataset_counts[ds_name] += 1
            mods = DATASET_MODALITIES[ds_name]

            # Helper to calculate and store for a specific mode (Base or TTA)
            def calc_sample_metrics(outputs, mode_name, modality_type):
                if outputs is None: return

                prefix = modality_type # 'he' or 'mif'

                # NUCLEI
                # Check if dataset wants this metric AND if output contains it
                if mods.get(f'{prefix}_nuclei', False):
                    if f'{prefix}_nuclei_seg' in outputs:
                        pred, _, _ = process_model_outputs(
                            outputs[f'{prefix}_nuclei_seg'][i, 0].cpu().numpy(),
                            outputs[f'{prefix}_nuclei_hv'][i, 0].cpu().numpy(),
                            outputs[f'{prefix}_nuclei_hv'][i, 1].cpu().numpy()
                        )
                        gt = batch[f'{prefix}_nuclei_instance'][i].cpu().numpy()
                        if gt.max() > 0:
                            pq, _, _ = get_fast_pq(gt, pred)
                            aji = aggregated_jaccard_index(gt, pred)
                            dataset_metrics[ds_name][mode_name][f'{prefix}_nuclei_pq'].append(pq)
                            dataset_metrics[ds_name][mode_name][f'{prefix}_nuclei_aji'].append(aji)

                # CELL
                if mods.get(f'{prefix}_cell', False):
                    if f'{prefix}_cell_seg' in outputs:
                        pred, _, _ = process_model_outputs(
                            outputs[f'{prefix}_cell_seg'][i, 0].cpu().numpy(),
                            outputs[f'{prefix}_cell_hv'][i, 0].cpu().numpy(),
                            outputs[f'{prefix}_cell_hv'][i, 1].cpu().numpy()
                        )
                        gt = batch[f'{prefix}_cell_instance'][i].cpu().numpy()
                        if gt.max() > 0:
                            pq, _, _ = get_fast_pq(gt, pred)
                            aji = aggregated_jaccard_index(gt, pred)
                            dataset_metrics[ds_name][mode_name][f'{prefix}_cell_pq'].append(pq)
                            dataset_metrics[ds_name][mode_name][f'{prefix}_cell_aji'].append(aji)

            # Store stats for Base
            calc_sample_metrics(out_he_base, 'base', 'he')
            if has_mif_in_batch: 
                calc_sample_metrics(out_mif_base, 'base', 'mif')
            
            # Store stats for TTA
            calc_sample_metrics(out_he_tta, 'tta', 'he')
            if has_mif_in_batch: 
                calc_sample_metrics(out_mif_tta, 'tta', 'mif')

# ==========================================
# 4. REPORTING (COMPARATIVE)
# ==========================================
print("\n" + "="*95)
print("📊 FINAL EVALUATION REPORT: BASE vs TTA")
print("="*95)

sorted_datasets = sorted(dataset_metrics.keys())

for ds_name in sorted_datasets:
    count = dataset_counts[ds_name]
    metrics = dataset_metrics[ds_name]
    
    print(f"\n📁 DATASET: {ds_name.upper()} (Tiles: {count})")
    print(f"{'Modality':<20} | {'Base PQ':<8} | {'TTA PQ':<8} | {'Δ PQ':<8} | {'Base AJI':<8} | {'TTA AJI':<8}")
    print("-" * 95)
    
    check_list = [
        ('H&E Nuclei', 'he_nuclei'),
        ('H&E Cells',  'he_cell'),
        ('MIF Nuclei', 'mif_nuclei'),
        ('MIF Cells',  'mif_cell')
    ]
    
    found_any = False
    for label, key in check_list:
        base_pq_list = metrics['base'].get(f'{key}_pq', [])
        tta_pq_list = metrics['tta'].get(f'{key}_pq', [])
        
        if len(base_pq_list) > 0:
            found_any = True
            b_pq = np.mean(base_pq_list)
            t_pq = np.mean(tta_pq_list)
            d_pq = t_pq - b_pq
            
            b_aji = np.mean(metrics['base'][f'{key}_aji'])
            t_aji = np.mean(metrics['tta'][f'{key}_aji'])
            
            sym = "+" if d_pq >= 0 else ""
            
            print(f"{label:<20} | {b_pq:.4f}   | {t_pq:.4f}   | {sym}{d_pq:.4f}   | {b_aji:.4f}     | {t_aji:.4f}")
            
    if not found_any:
        print("  (No valid metrics computed for this dataset)")

    print("-" * 95)

print("\n✅ All datasets processed.")

Using device: cuda

📦 Loading model...
✓ VitaminPFlex PRO initialized with large backbone
  + ASPP Bridge Active (Context Enhancement)
  + CoordConv Heads Active (Spatial Awareness)
✅ Model loaded

🚀 Starting Global Evaluation (Comparing Base vs. TTA)...


Processing Test Set: 100%|██████████| 171/171 [06:12<00:00,  2.18s/it]


📊 FINAL EVALUATION REPORT: BASE vs TTA

📁 DATASET: PANNUKE (Tiles: 681)
Modality             | Base PQ  | TTA PQ   | Δ PQ     | Base AJI | TTA AJI 
-----------------------------------------------------------------------------------------------
H&E Nuclei           | 0.8523   | 0.7916   | -0.0608   | 0.8828     | 0.8234
-----------------------------------------------------------------------------------------------

✅ All datasets processed.


## Whole dataet with STD

In [10]:
import torch
import numpy as np
from collections import defaultdict
from tqdm import tqdm
from metrics import get_fast_pq, aggregated_jaccard_index
from vitaminp import VitaminPFlex, SimplePreprocessing, prepare_he_input, prepare_mif_input
from postprocessing import process_model_outputs

# ==========================================
# 1. CONFIGURATION: DATASET MODALITIES
# ==========================================
DATASET_MODALITIES = {

    # MIF Only Datasets
    'tissuenet': {'he_nuclei': False, 'he_cell': False,  'mif_nuclei': True,  'mif_cell': True},

    # Combined / Paired Datasets
    'crc':       {'he_nuclei': True,  'he_cell': True,   'mif_nuclei': True,  'mif_cell': True},
    # H&E Only Datasets
    'pannuke':   {'he_nuclei': True,  'he_cell': False,   'mif_nuclei': False, 'mif_cell': False},
    'lizard':    {'he_nuclei': True,  'he_cell': False,  'mif_nuclei': False, 'mif_cell': False},
    'monuseg':   {'he_nuclei': True,  'he_cell': False,  'mif_nuclei': False, 'mif_cell': False},
    'tnbc':      {'he_nuclei': True,  'he_cell': False,  'mif_nuclei': False, 'mif_cell': False},
    'nuinsseg':  {'he_nuclei': True,  'he_cell': False,  'mif_nuclei': False, 'mif_cell': False},
    'cryonuseg': {'he_nuclei': True,  'he_cell': False,  'mif_nuclei': False, 'mif_cell': False},
    'bc':        {'he_nuclei': True,  'he_cell': False,  'mif_nuclei': False, 'mif_cell': False},
    'consep':    {'he_nuclei': True,  'he_cell': False,  'mif_nuclei': False, 'mif_cell': False},
    'monusac':   {'he_nuclei': True,  'he_cell': False,  'mif_nuclei': False, 'mif_cell': False},
    'kumar':     {'he_nuclei': True,  'he_cell': False,  'mif_nuclei': False, 'mif_cell': False},
    'cpm17':     {'he_nuclei': True,  'he_cell': False,  'mif_nuclei': False, 'mif_cell': False},
    

}

# ==========================================
# 2. SETUP & TTA HELPERS
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

print("\n📦 Loading model...")
model = VitaminPFlex(model_size='large').to(device)
checkpoint_path = "checkpoints/vitamin_p_flex_large_fold33_pannuke_lessaug_best.pth"
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()
preprocessor = SimplePreprocessing()
print(f"✅ Model loaded")

# Structure: metrics[dataset][mode][metric_name] -> list
dataset_metrics = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
dataset_counts = defaultdict(int)

def get_tta_outputs(model, img_tensor):
    """
    Runs inference 3 times (Orig, HFlip, VFlip) and returns 
    both the Original output (Base) and the Averaged output (TTA).
    """
    # 1. Original
    out_orig = model(img_tensor)

    # 2. H-Flip
    img_h = torch.flip(img_tensor, [3])
    out_h = model(img_h)
    # De-augment H
    for k in out_h.keys():
        out_h[k] = torch.flip(out_h[k], [3])
        if 'hv' in k: out_h[k][:, 0, :, :] *= -1 

    # 3. V-Flip
    img_v = torch.flip(img_tensor, [2])
    out_v = model(img_v)
    # De-augment V
    for k in out_v.keys():
        out_v[k] = torch.flip(out_v[k], [2])
        if 'hv' in k: out_v[k][:, 1, :, :] *= -1 

    # Aggregate
    out_avg = {}
    for k in out_orig.keys():
        stack = torch.stack([out_orig[k], out_h[k], out_v[k]])
        out_avg[k] = torch.mean(stack, dim=0)
        
    return out_orig, out_avg

# ==========================================
# 3. EVALUATION LOOP
# ==========================================
print("\n🚀 Starting Global Evaluation (Comparing Base vs. TTA)...")

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Processing Test Set"):
        
        sources = batch.get('dataset_source', [])
        
        # --- Prepare Inputs ---
        
        # 1. H&E Input (Prepare if ANY dataset needs it, or just prepare always to be safe/simple)
        # Optimization: We can check if any source needs H&E, but for now we run it standard.
        he_img = prepare_he_input(batch['he_image'].to(device))
        he_img = preprocessor.percentile_normalize(he_img)
        
        # 2. MIF Input (Check if ANY source in this batch requires MIF)
        has_mif_in_batch = False
        for s in sources:
            if s in DATASET_MODALITIES:
                if DATASET_MODALITIES[s]['mif_nuclei'] or DATASET_MODALITIES[s]['mif_cell']:
                    has_mif_in_batch = True
                    break
        
        mif_img = None
        if has_mif_in_batch:
            mif_img = prepare_mif_input(batch['mif_image'].to(device))
            mif_img = preprocessor.percentile_normalize(mif_img)
        
        # --- Run Inference (Get BOTH Base and TTA) ---
        out_he_base, out_he_tta = get_tta_outputs(model, he_img)
        
        out_mif_base, out_mif_tta = (None, None), (None, None)
        if has_mif_in_batch:
            out_mif_base, out_mif_tta = get_tta_outputs(model, mif_img)

        # Iterate samples in batch
        batch_size = len(sources)
        for i in range(batch_size):
            ds_name = sources[i]
            if ds_name not in DATASET_MODALITIES: continue
                
            dataset_counts[ds_name] += 1
            mods = DATASET_MODALITIES[ds_name]

            # Helper to calculate and store for a specific mode (Base or TTA)
            def calc_sample_metrics(outputs, mode_name, modality_type):
                if outputs is None: return

                prefix = modality_type # 'he' or 'mif'

                # NUCLEI
                # Check if dataset wants this metric AND if output contains it
                if mods.get(f'{prefix}_nuclei', False):
                    if f'{prefix}_nuclei_seg' in outputs:
                        pred, _, _ = process_model_outputs(
                            outputs[f'{prefix}_nuclei_seg'][i, 0].cpu().numpy(),
                            outputs[f'{prefix}_nuclei_hv'][i, 0].cpu().numpy(),
                            outputs[f'{prefix}_nuclei_hv'][i, 1].cpu().numpy()
                        )
                        gt = batch[f'{prefix}_nuclei_instance'][i].cpu().numpy()
                        if gt.max() > 0:
                            pq, _, _ = get_fast_pq(gt, pred)
                            aji = aggregated_jaccard_index(gt, pred)
                            dataset_metrics[ds_name][mode_name][f'{prefix}_nuclei_pq'].append(pq)
                            dataset_metrics[ds_name][mode_name][f'{prefix}_nuclei_aji'].append(aji)

                # CELL
                if mods.get(f'{prefix}_cell', False):
                    if f'{prefix}_cell_seg' in outputs:
                        pred, _, _ = process_model_outputs(
                            outputs[f'{prefix}_cell_seg'][i, 0].cpu().numpy(),
                            outputs[f'{prefix}_cell_hv'][i, 0].cpu().numpy(),
                            outputs[f'{prefix}_cell_hv'][i, 1].cpu().numpy()
                        )
                        gt = batch[f'{prefix}_cell_instance'][i].cpu().numpy()
                        if gt.max() > 0:
                            pq, _, _ = get_fast_pq(gt, pred)
                            aji = aggregated_jaccard_index(gt, pred)
                            dataset_metrics[ds_name][mode_name][f'{prefix}_cell_pq'].append(pq)
                            dataset_metrics[ds_name][mode_name][f'{prefix}_cell_aji'].append(aji)

            # Store stats for Base
            calc_sample_metrics(out_he_base, 'base', 'he')
            if has_mif_in_batch: 
                calc_sample_metrics(out_mif_base, 'base', 'mif')
            
            # Store stats for TTA
            calc_sample_metrics(out_he_tta, 'tta', 'he')
            if has_mif_in_batch: 
                calc_sample_metrics(out_mif_tta, 'tta', 'mif')

# ==========================================
# 4. REPORTING (COMPARATIVE)
# ==========================================
print("\n" + "="*130)
print("📊 FINAL EVALUATION REPORT: BASE vs TTA")
print("="*130)

sorted_datasets = sorted(dataset_metrics.keys())

for ds_name in sorted_datasets:
    count = dataset_counts[ds_name]
    metrics = dataset_metrics[ds_name]
    
    print(f"\n📁 DATASET: {ds_name.upper()} (Tiles: {count})")
    print(f"{'Modality':<20} | {'Base PQ':<15} | {'TTA PQ':<15} | {'Δ PQ':<8} | {'Base AJI':<15} | {'TTA AJI':<15}")
    print("-" * 130)
    
    check_list = [
        ('H&E Nuclei', 'he_nuclei'),
        ('H&E Cells',  'he_cell'),
        ('MIF Nuclei', 'mif_nuclei'),
        ('MIF Cells',  'mif_cell')
    ]
    
    found_any = False
    for label, key in check_list:
        base_pq_list = metrics['base'].get(f'{key}_pq', [])
        tta_pq_list = metrics['tta'].get(f'{key}_pq', [])
        
        if len(base_pq_list) > 0:
            found_any = True
            b_pq = np.mean(base_pq_list)
            b_pq_std = np.std(base_pq_list)
            t_pq = np.mean(tta_pq_list)
            t_pq_std = np.std(tta_pq_list)
            d_pq = t_pq - b_pq
            
            b_aji = np.mean(metrics['base'][f'{key}_aji'])
            b_aji_std = np.std(metrics['base'][f'{key}_aji'])
            t_aji = np.mean(metrics['tta'][f'{key}_aji'])
            t_aji_std = np.std(metrics['tta'][f'{key}_aji'])
            
            sym = "+" if d_pq >= 0 else ""
            
            print(f"{label:<20} | {b_pq:.4f}±{b_pq_std:.4f} | {t_pq:.4f}±{t_pq_std:.4f} | {sym}{d_pq:.4f}   | {b_aji:.4f}±{b_aji_std:.4f} | {t_aji:.4f}±{t_aji_std:.4f}")
            
    if not found_any:
        print("  (No valid metrics computed for this dataset)")

    print("-" * 130)

print("\n✅ All datasets processed.")

Using device: cuda

📦 Loading model...
✓ VitaminPFlex PRO initialized with large backbone
  + ASPP Bridge Active (Context Enhancement)
  + CoordConv Heads Active (Spatial Awareness)
✅ Model loaded

🚀 Starting Global Evaluation (Comparing Base vs. TTA)...


Processing Test Set: 100%|██████████| 553/553 [51:01<00:00,  5.54s/it]  


📊 FINAL EVALUATION REPORT: BASE vs TTA

📁 DATASET: BC (Tiles: 66)
Modality             | Base PQ         | TTA PQ          | Δ PQ     | Base AJI        | TTA AJI        
----------------------------------------------------------------------------------------------------------------------------------
H&E Nuclei           | 0.6268±0.1093 | 0.6367±0.1063 | +0.0099   | 0.6923±0.0833 | 0.7001±0.0831
----------------------------------------------------------------------------------------------------------------------------------

📁 DATASET: CONSEP (Tiles: 56)
Modality             | Base PQ         | TTA PQ          | Δ PQ     | Base AJI        | TTA AJI        
----------------------------------------------------------------------------------------------------------------------------------
H&E Nuclei           | 0.4936±0.0797 | 0.5140±0.0787 | +0.0204   | 0.5497±0.0751 | 0.5667±0.0692
-----------------------------------------------------------------------------------------------------------